# SPINE-GPE v7 — PNAD COVID Certification Engine v1.0.0
Certificação da ponte pandêmica sem identificação falsa de plataforma.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil
ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
SCRIPT_CANDIDATES = [Path("/content/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.0.py"), ROOT / "scripts/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.0.py"]
REQ_CANDIDATES = [Path("/content/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.0.txt"), ROOT / "scripts/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.0.txt"]
SCRIPT = next((p for p in SCRIPT_CANDIDATES if p.exists()), SCRIPT_CANDIDATES[0])
REQ = next((p for p in REQ_CANDIDATES if p.exists()), REQ_CANDIDATES[0])
print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("REQ:", REQ, REQ.exists())
if not SCRIPT.exists(): raise FileNotFoundError(f"Script não encontrado: {SCRIPT_CANDIDATES}")
if not REQ.exists(): raise FileNotFoundError(f"Requirements não encontrado: {REQ_CANDIDATES}")


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","--prefer-binary","-r",str(REQ)],check=True)
subprocess.run([sys.executable,"-m","py_compile",str(SCRIPT)],check=True)
print("py_compile: OK")


## Auditoria — setembro de 2020

In [ ]:
audit = subprocess.run([sys.executable,str(SCRIPT),"--root",str(ROOT),"--mode","audit","--months","9","--download","--strict"],text=True,capture_output=True)
print(audit.stdout); print(audit.stderr); print("Exit code:",audit.returncode)


## Certificação — setembro de 2020

In [ ]:
cert = subprocess.run([sys.executable,str(SCRIPT),"--root",str(ROOT),"--mode","certify","--months","9","--download","--chunk-rows","50000","--strict"],text=True,capture_output=True)
print(cert.stdout); print(cert.stderr); print("Exit code:",cert.returncode)


In [ ]:
LOCK = ROOT / "00_admin/PNAD_COVID_CERTIFICATION_LOCK.json"
lock=json.loads(LOCK.read_text(encoding="utf-8"))
print(json.dumps(lock,ensure_ascii=False,indent=2))
assert lock["status"] in {"CERTIFIED","CORE_CERTIFIED"}, lock["critical_failures"]


## Série completa maio–novembro (após validar setembro)

In [ ]:
# full = subprocess.run([sys.executable,str(SCRIPT),"--root",str(ROOT),"--mode","certify","--months","all","--download","--chunk-rows","50000","--strict"],check=False)
